# VPRL Parameter Recovery

#coded/run by Ashley Shipp (May 2026)

## Import

In [ ]:
import os
import numpy as np
import random
import csv
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from cmdstanpy import CmdStanModel
os.environ["CXX"] = "g++"
os.environ["TBB_CXX_TYPE"] = "gcc"
from simulate import doVPRL  

# Compile once!
VPRLmodel = CmdStanModel(stan_file="G:/My Drive/Research/Code/Stan Parameter Recovery/VPRL.stan")
print("Compile finished!")

## Parameters

In [ ]:
#test run examples to ensure accurate recovery
simulation_parameter_sets = {
    'group_names': ["post_ect", "ect_v2"],
    'alphaRewards': [.31, .31],
    'gammaRewards': [.54, .54],
    'alphaPunishments': [.25, .25],
    'gammaPunishments': [.25, .25],
    'taus': [.08,.08]
}

# run all groups
# 
# # simulation_parameter_sets = {
#     'group_names': ["pre_ect", "post_ect", "non_ect_v1", "non_ect_v2", "no_depression_v1", "no_depression_v2"],
#     'alphaRewards': [0.15, 0.31, 0.21, 0.12, 0.16, 0.18],
#     'gammaRewards': [0.70, 0.54, 0.83, 0.79, 0.83, 0.73],
#     'alphaPunishments': [0.45, 0.25, 0.46, 0.52, 0.53, 0.41],
#     'gammaPunishments': [0.23, 0.25, 0.11, 0.15, 0.19, 0.23],
#     'taus': [0.10, 0.08, 0.17, 0.08, 0.11, 0.07]
# }

num_simulated_subjects = 50
iter_sampling=8000
iter_warmup=2000

trials_p1 = 25
trials_p2 = 50
trials_p3 = 75
total_trials = trials_p1 + trials_p2 + trials_p3
num_icons = 6

arms = {
    0: [(0.25, 1.0),  (0.25, 1.0),  (0.25, 2.5)],
    1: [(0.50, 1.0),  (0.50, 1.0),  (0.50, 1.5)],
    2: [(0.75, 1.0),  (0.75, 1.0),  (0.75, 0.5)],
    3: [(0.0,  0.0),  (0.25, -1.0), (0.25, -1.25)], 
    4: [(0.0,  0.0),  (0.50, -1.0), (0.50, -0.75)],
    5: [(0.0,  0.0),  (0.75, -1.0), (0.75, -0.25)]
}

print("Parameters set!")

## Simulation Loop

In [ ]:
for g_idx in range(len(simulation_parameter_sets['group_names'])):
    group_name = simulation_parameter_sets['group_names'][g_idx]
    
    print(f"\n\n{'='*70}")
    print(f"PROCESSING GROUP: {group_name.upper()}")
    print(f"{'='*70}")
    
    # Extract current group parameters
    curr_alphaReward = simulation_parameter_sets['alphaRewards'][g_idx]
    curr_gammaReward = simulation_parameter_sets['gammaRewards'][g_idx]
    curr_alphaPunishment = simulation_parameter_sets['alphaPunishments'][g_idx]
    curr_gammaPunishment = simulation_parameter_sets['gammaPunishments'][g_idx]
    curr_tau = simulation_parameter_sets['taus'][g_idx]

    simulated_data = []

    # -------------------------------------------------------------------------
    # A. Generate Data
    # -------------------------------------------------------------------------
    for sub in range(num_simulated_subjects):
        stuff = {
            'totalNumOfTrials': total_trials,
            'fullIconDisplayIndex': [],
            'taskValueStructure': [[0.0] * total_trials for _ in range(num_icons * 2)], 
            'allVPRLforGraphing': [],
            # ---> DYNAMIC INJECTION <--
            'tau': curr_tau,
            'alphaReward': curr_alphaReward,
            'alphaPunishment': curr_alphaPunishment,
            'gammaReward': curr_gammaReward,
            'gammaPunishment': curr_gammaPunishment
        }

        for t in range(total_trials):
            if t < trials_p1:
                phase_idx = 0
                valid_icons = [0, 1, 2]
            elif t < trials_p1 + trials_p2:
                phase_idx = 1
                valid_icons = [0, 1, 2, 3, 4, 5]
            else:
                phase_idx = 2
                valid_icons = [0, 1, 2, 3, 4, 5]

            o1, o2 = random.sample(valid_icons, 2)
            stuff['fullIconDisplayIndex'].append([o1, o2])

            for icon in range(num_icons):
                prob, amt = arms[icon][phase_idx]
                is_win = np.random.rand() < prob
                
                r, p = 0.0, 0.0
                if is_win:
                    if amt > 0:
                        r = amt
                    elif amt < 0:
                        p = abs(amt) 
                        
                stuff['taskValueStructure'][icon][t] = p               
                stuff['taskValueStructure'][icon + num_icons][t] = r   

        stuff, totalWinnings = doVPRL(stuff)
        
        choice_array = stuff['choiceArrayVPRL']
        sub_opt1, sub_opt2, sub_choice, sub_rew, sub_pun = [], [], [], [], []
        
        for t in range(total_trials):
            left_icon = stuff['fullIconDisplayIndex'][t][0]
            right_icon = stuff['fullIconDisplayIndex'][t][1]
            chosen_side = choice_array[t] 
            
            sub_opt1.append(left_icon + 1)
            sub_opt2.append(right_icon + 1)
            sub_choice.append(chosen_side + 1) 
            
            chosen_icon = left_icon if chosen_side == 0 else right_icon
            r = stuff['taskValueStructure'][chosen_icon + num_icons][t]
            p = stuff['taskValueStructure'][chosen_icon][t]
            
            sub_rew.append(r)
            sub_pun.append(p)
            
        simulated_data.append({
            'option1': str(sub_opt1),
            'option2': str(sub_opt2),
            'choice': str(sub_choice),
            'reward': str(sub_rew),
            'punish': str(sub_pun),
            'tau': curr_tau,
            'alphaReward': curr_alphaReward,
            'alphaPunishment': curr_alphaPunishment,
            'gammaReward': curr_gammaReward,
            'gammaPunishment': curr_gammaPunishment
        })

    # Save Dynamic CSV
    csv_file = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/generated_data_VPRL_{group_name}.csv'
    with open(csv_file, 'w', newline='') as csvfile:
        fieldnames = ['option1', 'option2', 'choice', 'reward', 'punish','tau','alphaReward','alphaPunishment','gammaReward','gammaPunishment']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in simulated_data:
            writer.writerow(row)
    print(f"Data generation complete for {group_name}. Saved to: {csv_file}")
    
    # -------------------------------------------------------------------------
    # B. Open .csv and import raw data
    # -------------------------------------------------------------------------
    with open(csv_file, newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        option1, option2, choice, reward, punish, tau, alphaReward, alphaPunishment, gammaReward, gammaPunishment = [], [], [], [], [], [], [], [], [], []
        for row in reader:
            option1.append(row['option1'].replace("[","").replace(" ","").replace("]","").split(","))
            option2.append(row['option2'].replace("[","").replace(" ","").replace("]","").split(","))
            choice.append(row['choice'].replace("[","").replace(" ","").replace("]","").split(","))
            reward.append(row['reward'].replace("[","").replace(" ","").replace("]","").split(","))
            punish.append(row['punish'].replace("[","").replace(" ","").replace("]","").split(","))
            tau.append(row['tau'].replace("[","").replace(" ","").replace("]","").split(","))
            alphaReward.append(row['alphaReward'].replace("[","").replace(" ","").replace("]","").split(","))
            alphaPunishment.append(row['alphaPunishment'].replace("[","").replace(" ","").replace("]","").split(","))
            gammaReward.append(row['gammaReward'].replace("[","").replace(" ","").replace("]","").split(","))
            gammaPunishment.append(row['gammaPunishment'].replace("[","").replace(" ","").replace("]","").split(","))

    num_subjects = len(option1)
    num_trials = len(option1[0])
    trials_per_subject = [num_trials for x in range(num_subjects)]

    outcome = []
    for r_row, p_row in zip(reward, punish):
        sub_outcome = []
        for r, p in zip(r_row, p_row):
            sub_outcome.append(float(r) - float(p)) 
        outcome.append(sub_outcome)

    raw_dat = {
        'N': num_subjects, 
        'T': num_trials, 
        'T_subjs': trials_per_subject, 
        'option1':[[int(i) for i in x] for x in option1], 
        'option2':[[int(i) for i in x] for x in option2], 
        'choice':[[int(i) for i in x] for x in choice], 
        'reward':[[float(i) for i in x] for x in reward], 
        'punish':[[float(i) for i in x] for x in punish], 
        'outcome': outcome,
    }

    # -------------------------------------------------------------------------
    # C. Fit data to model
    # -------------------------------------------------------------------------
    print(f"\nSampling data from model for {group_name}...")

    fit = VPRLmodel.sample(data=raw_dat, iter_sampling=iter_sampling, iter_warmup=iter_warmup, chains=2, show_progress=True, adapt_delta=0.95)

    # Save sampled data to CSVs
    draws_df = fit.draws_pd()
    summary_df = draws_df.describe(percentiles=[0.025, 0.50, 0.975]).T # Ensure 50% median is caught
    
    draws_path = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/all_modeled_data_VPRL_{group_name}.csv'
    summary_path = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/modeled_data_VPRL_{group_name}.csv'
    draws_df.to_csv(draws_path)
    summary_df.to_csv(summary_path)

    modeled_data_import = {}
    for key, row in summary_df.iterrows():
        modeled_data_import[key] = {
            "mean": row['mean'],
            "median": row['50%'], 
            "sd": row['std'],
            "errbar_low": row['2.5%'],
            "errbar_high": row['97.5%']
        }
    print(f"Exported fit data for {group_name}!")

    # -------------------------------------------------------------------------
    # D. PLOT 1: Recovered Parameter Posterior Densities
    # -------------------------------------------------------------------------
    variables = ["learnrate_pos", "discount_pos", "learnrate_neg", "discount_neg", "tau"]
    colors = ['darkorchid', 'plum', 'navy', 'steelblue', 'seagreen']

    labels = [
        "Reward Learning Rate", 
        "Reward Discount Factor", 
        "Punishment Learning Rate", 
        "Punishment Discount Factor", 
        "Inverse Temperature (1/Tau)"
    ]

    # ---> DYNAMIC INJECTION: Use current loop's targets <---
    actuals = [curr_alphaReward, curr_gammaReward, curr_alphaPunishment, curr_gammaPunishment, curr_tau]

    f = plt.figure()
    f.set_figwidth(35)
    f.set_figheight(5)
    bins = 100

    for i in range(len(variables)):
        var = variables[i]
        color = colors[i]
        
        data = draws_df[f'mu_{var}']
        
        mean_val = data.mean()
        sd_val = data.std()
        
        x_axis = np.linspace(min(data)-sd_val, max(data)+sd_val, 100)
        p_val = norm.pdf(x_axis, mean_val, sd_val)
        
        plt.hist(data, bins, density=True, alpha=0.5, color=color)
        plt.plot(x_axis, p_val, color=color, linewidth=2)
        plt.axvline(x=actuals[i], color=color, linewidth=5, linestyle="solid")

    plt.xlim(0, 1)
    plt.xticks(fontsize=24)
    plt.yticks([])

    plt.xlabel('Value', fontsize=24)
    plt.ylabel('Probability Density', fontsize=24)
    
    # ---> DYNAMIC INJECTION: Title includes the group name <---
    plt.title(f"Recovered Parameter Posterior Densities ({group_name.upper()})", fontsize=30, pad=15)

    legend_handles = []

    for i in range(len(variables)):
        legend_handles.append(mpatches.Patch(color=colors[i], alpha=0.5, label=labels[i]))

    legend_handles.append(mlines.Line2D([], [], color="black", linewidth=5, linestyle="solid", label="Actual Target"))

    plt.legend(handles=legend_handles, bbox_to_anchor=(1.00, 1), loc='upper right', fontsize=18)
    
    # ---> DYNAMIC INJECTION: Save Plot 1 <---
    plot1_path = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/Posterior_Densities_{group_name}.png'
    plt.savefig(plot1_path, bbox_inches='tight')
    plt.show()

    # -------------------------------------------------------------------------
    # E. PLOT 2: Individual Agent Recovery Grid
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(25, 10), sharex='col', gridspec_kw={'height_ratios': [1, 4]})
    plt.subplots_adjust(hspace=0.05, wspace=0.05, right=0.88) 

    variables_names = ["Reward\nLearning Rate", "Reward\nDiscount Factor", "Punishment\nLearning Rate", "Punishment\nDiscount Factor", "Inverse Temperature\n(1/Tau)"]
    actuals_for_plot = actuals

    for col, var in enumerate(variables):
        ax_top = axes[0, col]
        ax_bot = axes[1, col]
        true_target = actuals_for_plot[col]
        color = colors[col]
        
        # --- TOP ROW: Group Posterior Histograms ---
        mu_data = draws_df[f'mu_{var}']
        counts, bins_plot, _ = ax_top.hist(mu_data, bins=50, color=color, alpha=0.5)
        
        recovered_group_mean = float(modeled_data_import[f'mu_{var}']['mean'])
        recovered_group_median = float(modeled_data_import[f'mu_{var}']['median'])
        
        # YOUR ORIGINAL COLOR CODING RESTORED
        ax_top.vlines(true_target, ymin=0, ymax=max(counts), color=colors[col], linestyle='solid', linewidth=4)
        ax_top.axvline(x=recovered_group_mean, color=colors[col], linestyle="dotted", linewidth=4)
        ax_top.axvline(x=recovered_group_median, color=colors[col], linestyle="dashed", linewidth=4)
        
        ax_top.set_title(variables_names[col], fontsize=16, pad=15)
        
        ax_top.tick_params(labelbottom=False)
        if col == 0:
            ax_top.set_ylabel("Probability Density", fontsize=14)
        else:
            ax_top.set_yticks([]) 

        # --- BOTTOM ROW: Individual Agent HDIs ---
        for x in range(1, num_subjects + 1):
            sim_key = f"{var}[{x}]"
            if sim_key in modeled_data_import:
                m = modeled_data_import[sim_key]['mean']
                l = modeled_data_import[sim_key]['errbar_low']
                h = modeled_data_import[sim_key]['errbar_high']
                
                c = colors[col] if l <= true_target <= h else "red"
                
                ax_bot.hlines(y=x, xmin=l, xmax=h, color=c, alpha=0.8, linewidth=3)
                ax_bot.plot(m, x, marker="o", mfc=c, mec=c, markersize=10, alpha=0.8)

        # YOUR ORIGINAL COLOR CODING RESTORED
        ax_bot.axvline(x=true_target, color=colors[col], linestyle="solid", linewidth=4)

        ax_bot.set_ylim(0, num_subjects + 1)
        ax_bot.set_xlabel('Value', fontsize=14)
        if col == 0:
            ax_bot.set_ylabel('Simulated Agent', fontsize=14)
        else:
            ax_bot.set_yticks([]) 

    # -------------------------------------------------------------------------
    # Master Figure Legend
    # -------------------------------------------------------------------------
    hist_patch = mpatches.Patch(color='black', 
                                alpha=0.5, label='Recovered Posterior')
    hist_true = mlines.Line2D([], [], color='black', linestyle='solid', linewidth=4, label='Actual Target')

    hdi_pass = mlines.Line2D([], [], color='black', marker='o', markersize= 10, linewidth=3, mfc='black', linestyle='solid', label='Agent HDI (Contains Target)')
    hdi_fail = mlines.Line2D([], [], color='red', marker='o', markersize=10, linewidth=3, mfc='red', linestyle='solid', label='Agent HDI (Misses Target)')

    bot_mean = mlines.Line2D([], [], color='black', linestyle='dotted', linewidth=4, label='Recovered Group Mean')
    bot_median = mlines.Line2D([], [], color='black', linestyle='dashed', linewidth=4, label='Recovered Group Median')

    # ---> DYNAMIC INJECTION: Title includes the group name <---
    fig.legend(handles=[hist_true, hist_patch, bot_mean, bot_median, hdi_pass, hdi_fail], 
               loc='upper right', 
               bbox_to_anchor=(1.005, 0.85), 
               fontsize=12, 
               frameon=True, 
               title=f"Legend ({group_name})", 
               title_fontsize=14)

    plt.rcParams['figure.dpi'] = 150
    
    # ---> DYNAMIC INJECTION: Save Plot 2 <---
    plot2_path = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/Agent_Recovery_Grid_{group_name}.png'
    plt.savefig(plot2_path, bbox_inches='tight')
    plt.show()
    
    print(f"[{group_name.upper()}] SIMULATIONS COMPLETE!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.ticker as ticker

# ==========================================
# DEFINE DESIRED GROUPS, PATHS & ACTUALS
# ==========================================
base_path = 'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups'

simulation_parameter_sets = {
    'group_names': ["pre_ect", "post_ect", "non_res_v1", "non_res_v2", "non_ect_v1", "non_ect_v2", "no_depression_v1", "no_depression_v2"],
    'group_titles': ["ECT Responders\nPre-ECT (Visit 1)", "ECT Responders\nPost-ECT (Visit 2)", "ECT Non-responders\nPre-ECT (Visit 1)", "ECT Non-responders\nPost-ECT (Visit 2)", "Non-ECT\n(Visit 1)", "Non-ECT\n(Visit 2)", "No-depression\n(Visit 1)", "No-depression\n(Visit 2)"],
    'alphaRewards': [0.15, 0.31, 0.12, 0.22, 0.21, 0.12, 0.16, 0.18],
    'gammaRewards': [0.70, 0.54, 0.72, 0.81, 0.83, 0.79, 0.83, 0.73],
    'alphaPunishments': [0.45, 0.25, 0.28, 0.42, 0.46, 0.52, 0.53, 0.41],
    'gammaPunishments': [0.23, 0.25, 0.34, 0.15, 0.11, 0.15, 0.19, 0.23],
    'taus': [0.10, 0.08, 0.08, 0.08, 0.17, 0.08, 0.11, 0.07]
}

# Define core structure variables mapping
variables = ["learnrate_pos", "discount_pos", "learnrate_neg", "discount_neg", "tau"]

variables_names = [ 
    "Positive System\nLearning Rate" + r"($\alpha^P$)",
    "Positive System\nDiscount Factor" + r"($\gamma^P$)",
    "Negative System\nLearning Rate" + r"($\alpha^N$)",
    "Negative System\nDiscount Factor" + r"($\gamma^N$)",
    "Inverse\nTemperature" + r"($1/\tau$)"
]
# variables_names = [ 
#     "Learning Rate" + r"($\alpha^P$)",
#     "Discount Factor" + r"($\gamma^P$)",
#     "Learning Rate" + r"($\alpha^N$)",
#     "Discount Factor" + r"($\gamma^N$)",
#     "Inverse\nTemperature" + r"($1/\tau$)"
# ]

colors = ['darkorchid', 'plum', 'navy', 'steelblue', 'seagreen']

# Loop through and generate only Plot 2 for selected groups
for idx,group_name in enumerate(simulation_parameter_sets['group_names']):
    group_title = simulation_parameter_sets['group_titles'][idx]
    print(f"\nProcessing Plot 2 for Group: {group_name.upper()}")
    
    # Load data files
    draws_path = f'{base_path}/all_modeled_data_VPRL_{group_name}.csv'
    summary_path = f'{base_path}/modeled_data_VPRL_{group_name}.csv'
    
    try:
        draws_df = pd.read_csv(draws_path)
        summary_df = pd.read_csv(summary_path, index_col=0) 
    except FileNotFoundError as e:
        print(f"Skipping group '{group_name}'. Ensure modeled CSV files exist.")
        continue

    # Extract static target values for the current group
    g_idx = simulation_parameter_sets['group_names'].index(group_name)
    curr_alphaReward = simulation_parameter_sets['alphaRewards'][g_idx]
    curr_gammaReward = simulation_parameter_sets['gammaRewards'][g_idx]
    curr_alphaPunishment = simulation_parameter_sets['alphaPunishments'][g_idx]
    curr_gammaPunishment = simulation_parameter_sets['gammaPunishments'][g_idx]
    curr_tau = simulation_parameter_sets['taus'][g_idx]
    
    actuals = [curr_alphaReward, curr_gammaReward, curr_alphaPunishment, curr_gammaPunishment, curr_tau]

    # Reconstruct modeled data dictionary structure safely
    modeled_data_import = {}
    for key, row in summary_df.iterrows():
        modeled_data_import[str(key)] = {
            "mean": row['mean'],
            "median": row['50%'], 
            "sd": row['std'],
            "errbar_low": row['2.5%'],
            "errbar_high": row['97.5%']
        }
        
    # Dynamically determine num_subjects
    num_subjects = sum(1 for key in modeled_data_import.keys() if str(key).startswith(f'{variables[0]}['))

    # Expanded figsize to account for giant text elements
    fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(36, 18), sharex='col', gridspec_kw={'height_ratios': [1, 4]})
    plt.subplots_adjust(hspace=0.12, wspace=0.12, right=0.78, left=0.08, bottom=0.10, top=0.90) 

    for col, var in enumerate(variables):
        ax_top = axes[0, col]
        ax_bot = axes[1, col]
        true_target = actuals[col]
        color = colors[col]
        
        # --- TOP ROW: Group Posterior Histograms ---
        mu_data = draws_df[f'mu_{var}']
        counts, bins_plot, _ = ax_top.hist(mu_data, bins=50, color=color, alpha=0.5, density=True)
        
        recovered_group_mean = float(modeled_data_import[f'mu_{var}']['mean'])
        recovered_group_median = float(modeled_data_import[f'mu_{var}']['median'])
        
        if len(counts) > 0:
            ax_top.vlines(true_target, ymin=0, ymax=max(counts), color=colors[col], linestyle='solid', linewidth=6)
            
        ax_top.axvline(x=recovered_group_mean, color=colors[col], linestyle="dotted", linewidth=6)
        ax_top.axvline(x=recovered_group_median, color=colors[col], linestyle="dashed", linewidth=6)
        
        # Giant Column Titles
        ax_top.set_title(variables_names[col], fontsize=36, pad=25)
        ax_top.tick_params(labelbottom=False)
        
        if col == 0:
            ax_top.set_ylabel("Density", fontsize=36, labelpad=15)
            ax_top.tick_params(axis='y', labelsize=32)
        else:
            ax_top.set_yticks([]) 

        # --- BOTTOM ROW: Individual Agent HDIs ---
        for x in range(1, num_subjects + 1):
            sim_key = f"{var}[{x}]"
            if sim_key in modeled_data_import:
                u = modeled_data_import[sim_key]['mean']
                m = modeled_data_import[sim_key]['median']
                l = modeled_data_import[sim_key]['errbar_low']
                h = modeled_data_import[sim_key]['errbar_high']
                
                c = colors[col] if l <= true_target <= h else "red"
                
                # Thickened lines and larger markers to match font weight
                ax_bot.hlines(y=x, xmin=l, xmax=h, color=c, alpha=0.8, linewidth=5)
                ax_bot.plot(m, x, marker="o", mfc=c, mec=c, markersize=15, alpha=0.8) #plot median filled circles
                ax_bot.plot(u, x, marker="o", mfc="none", mec=c, markersize=15, alpha=0.8) #plot mean hollow circles


        ax_bot.axvline(x=true_target, color=colors[col], linestyle="solid", linewidth=6)
        ax_bot.set_ylim(0, num_subjects + 1)
        
        # Giant X-Axis Labels
        ax_bot.set_xlabel('Value', fontsize=36, labelpad=15)
        
        # CRITICAL FIX: Keep all physical tick marks, but only label every other one (even positions)
        ax_bot.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:g}" if pos % 2 == 0 else ""))
        ax_bot.tick_params(axis='x', labelsize=32, pad=12)
        
        if col == 0:
            ax_bot.set_ylabel('Simulated Agent', fontsize=36, labelpad=15)
            ax_bot.tick_params(axis='y', labelsize=32)
        else:
            ax_bot.set_yticks([]) 
        
        # --- ADD OVERARCHING SYSTEM TITLES ---
        # Get physical bounding boxes of the top row axes
        pos0 = axes[0, 0].get_position()
        pos1 = axes[0, 1].get_position()
        pos2 = axes[0, 2].get_position()
        pos3 = axes[0, 3].get_position()

        # Calculate exact horizontal midpoints across the paired columns
        center_pos = (pos0.x0 + pos1.x1) / 2
        center_neg = (pos2.x0 + pos3.x1) / 2

        # # Place text in the top margin (y=0.95 sits safely above the top=0.90 subplot boundary)
        # fig.text(center_pos, 0.95, "Positive System", ha='center', va='bottom', fontsize=40)
        # fig.text(center_neg, 0.95, "Negative System", ha='center', va='bottom', fontsize=40)

    # --- Master Figure Legend (Enlarged Elements) ---
    hist_patch = mpatches.Patch(color='black', alpha=0.5, label='Recovered Posterior')
    hist_true = mlines.Line2D([], [], color='black', linestyle='solid', linewidth=5, label='Actual Target')
    hdi_pass = mlines.Line2D([], [], color='black', marker='o', markersize=14, linewidth=4, mfc='black', linestyle='solid', label='Agent HDI (Contains Target)')
    hdi_fail = mlines.Line2D([], [], color='red', marker='o', markersize=14, linewidth=4, mfc='red', linestyle='solid', label='Agent HDI (Misses Target)')
    bot_mean = mlines.Line2D([], [], color='black', linestyle='dotted', linewidth=5, label='Recovered Group Mean')
    bot_median = mlines.Line2D([], [], color='black', linestyle='dashed', linewidth=5, label='Recovered Group Median')
    cir_med = mlines.Line2D([], [], color='black', marker='o', markersize=14, mfc='black', linestyle='none', label='Agent Median')
    cir_mean = mlines.Line2D([], [], color='black', marker='o', markersize=14, mfc='none', linestyle='none', label='Agent Mean')


    leg = fig.legend(
        handles=[hist_true, hist_patch, bot_mean, bot_median, hdi_pass, hdi_fail, cir_med, cir_mean], 
        loc='center right', 
        bbox_to_anchor=(0.995, 0.5), 
        fontsize=30, 
        frameon=True, 
        title=group_title, 
        title_fontproperties={'weight': 'bold', 'size': 34},
        alignment='left'
    )

    plt.rcParams['figure.dpi'] = 150
    
    # Save step
    plot2_path = f'{base_path}/New_Agent_Recovery_Grid_{group_name}.png'
    plt.savefig(plot2_path, bbox_inches='tight', dpi=150)
    plt.show()
    
    print(f"Plot successfully saved with double-scale fonts to: {plot2_path}")

## Table S1

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import dataframe_image as dfi
import re
from html2image import Html2Image
from PIL import Image, ImageChops
import os

# Mapping requested group names to their exact file names and true generating parameters
group_info = {
    ("ECT\nResponders","(Visit 1)"):    {"file": "pre_ect",          "targets": [0.15, 0.70, 0.45, 0.23, 0.10]},
    ("ECT\nResponders","(Visit 2)"):   {"file": "post_ect",         "targets": [0.31, 0.54, 0.25, 0.25, 0.08]},
    ("ECT\nNon-responders","(Visit 1)"): {"file": "non_res_v1",       "targets": [0.12, 0.72, 0.28, 0.34, 0.08]},
    ("ECT\nNon-responders","(Visit 2)"): {"file": "non_res_v2",       "targets": [0.22, 0.81, 0.42, 0.15, 0.08]},
    ("Non-ECT","(Visit 1)"): {"file": "non_ect_v1",       "targets": [0.21, 0.83, 0.46, 0.11, 0.17]},
    ("Non-ECT","(Visit 2)"): {"file": "non_ect_v2",       "targets": [0.12, 0.79, 0.52, 0.15, 0.08]},
    ("No-depression","(Visit 1)"):  {"file": "no_depression_v1", "targets": [0.16, 0.83, 0.53, 0.19, 0.11]},
    ("No-depression","(Visit 2)"):  {"file": "no_depression_v2", "targets": [0.18, 0.73, 0.41, 0.23, 0.07]},
}

variables = ["learnrate_pos", "discount_pos", "learnrate_neg", "discount_neg", "tau"]

# Native HTML representations (renders instantly, no MathJax required)
var_names_latex = {
    "learnrate_pos": "α<sup>P</sup>",
    "discount_pos": "γ<sup>P</sup>",
    "learnrate_neg": "α<sup>N</sup>",
    "discount_neg": "γ<sup>N</sup>",
    "tau": "1/τ"
}

num_subjects = 50
results_list = []

print("Calculating recovery metrics across all groups...")

for group_name, info in group_info.items():
    file_path = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/modeled_data_VPRL_{info["file"]}.csv'
    
    try:
        df = pd.read_csv(file_path, index_col=0)
    except FileNotFoundError:
        print(f"Warning: Could not find data for {group_name} at {file_path}. Skipping.")
        continue

    for col_idx, var in enumerate(variables):
        true_target = info["targets"][col_idx]
        latex_var = var_names_latex[var]
        
        group_sd = df.loc[f'mu_{var}', 'std']
        group_mu = df.loc[f'mu_{var}', 'mean']
        
        agent_shrinkages = []
        agent_zscores = []
        agent_pvals = []
        agent_kls = []
        hdi_passes = []
        
        for x in range(1, num_subjects + 1):
            sim_key = f"{var}[{x}]"
            
            if sim_key in df.index:
                agent_mean = df.loc[sim_key, 'mean']
                agent_sd = df.loc[sim_key, 'std']
                agent_low = df.loc[sim_key, '2.5%']
                agent_high = df.loc[sim_key, '97.5%']
                
                in_hdi = 1 if (agent_low <= true_target <= agent_high) else 0
                hdi_passes.append(in_hdi)
                
                shrinkage = 1 - ((agent_sd**2) / (group_sd**2))
                agent_shrinkages.append(shrinkage)
                
                kl_div = np.log(group_sd / agent_sd) + ((agent_sd**2 + (agent_mean - group_mu)**2) / (2 * group_sd**2)) - 0.5
                agent_kls.append(kl_div)
                
                z_score = np.abs(agent_mean - true_target) / agent_sd
                agent_zscores.append(z_score)
                
                p_val = 2 * norm.sf(z_score)
                agent_pvals.append(p_val)
                
        results_list.append({
            "Main_Group": group_name[0],
            "Sub_Group": group_name[1],
            "Parameter": latex_var,
            "Percent in\n95% HDI": np.mean(hdi_passes) * 100,
            "Posterior\nShrinkage": np.mean(agent_shrinkages),
            "KLD": np.mean(agent_kls),
            "Z-Score": np.mean(agent_zscores),
            "Percent\nZ<1.96": np.mean(np.array(agent_pvals) > 0.05) * 100 
        })

# =========================================================================
# Create DataFrame and Format
# =========================================================================
metrics_df = pd.DataFrame(results_list)

# Now using a 3-level index!
styled_df = metrics_df.set_index(['Main_Group', 'Sub_Group', 'Parameter'])
styled_df.index.names = [None, None, None]

format_dict = {
    'Percent in\n95% HDI': '{:.0f}',
    'Posterior\nShrinkage': '{:.3f}', 
    'KLD': '{:.3f}',
    'Z-Score': '{:.3f}', 
    'Percent\nZ<1.96': '{:.0f}'
}

table_styles = [
    # 1. Main Group Column (Spans 10 rows)
    {'selector': '.row_heading.level0', 'props': [
        ('writing-mode', 'vertical-rl'),
        ('transform', 'rotate(180deg)'),
        ('text-align', 'center'),
        ('vertical-align', 'middle'),
        ('font-weight', 'bold'),
        ('font-size', '26px'),          
        ('letter-spacing', '2px'),      
        ('border-right', '1px solid black'),
        ('border-left', '3px solid black'),
        ('border-bottom','3px solid black'),
        ('border-top','3px solid black'),
        ('padding', '10px'),
        ('white-space','pre-wrap')
    ]},
    # 2. Sub Group Column (Spans 5 rows)
    {'selector': '.row_heading.level1', 'props': [
        ('writing-mode', 'vertical-rl'),
        ('transform', 'rotate(180deg)'),
        ('text-align', 'center'),
        ('vertical-align', 'middle'),
        ('font-weight', 'bold'),
        ('font-size', '22px'),          
        ('border-right', '1px solid black'),
        ('border-bottom','3px solid black'),
        ('border-top','3px solid black'),
        ('padding', '10px'),
        ('white-space','pre-wrap')
    ]},
    # 3. Parameter Symbols (Now level2)
    {'selector': '.row_heading.level2', 'props': [
        ('text-align', 'center'),
        ('vertical-align', 'middle'),
        ('font-size', '26px'),
        ('font-weight', 'normal'),
        ('padding', '6px 15px')
    ]},
    # 4. Main Data Headers
    {'selector': 'th.col_heading', 'props': [
        ('border-bottom', '3px solid black'),
        ('text-align', 'center'),
        ('vertical-align', 'bottom'), 
        ('font-size', '22px'), 
        ('white-space', 'pre-wrap'), 
        ('padding-bottom', '10px'),
        ('padding', '8px')
    ]},
    # 5. Hide the blank index headers
    {'selector': 'th.index_name', 'props': [
        ('display', 'none')
    ]},
    # 6. Bottom borders for data cells
    {'selector': 'tbody tr:nth-child(5n) td', 'props': [
        ('border-bottom', '3px solid black')
    ]},
    # 7. Bottom borders for parameter cells
    {'selector': 'tbody tr:nth-child(5n) th.level2', 'props': [
        ('border-bottom', '3px solid black')
    ]},
    # 8. Entire table border
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('border', '3px solid black')
    ]}
]

final_styled_table = (styled_df.style.format(format_dict)
                        .set_table_styles(table_styles)
                        .set_properties(**{
                            'border': '1px solid #e0e0e0', 
                            'padding': '8px 10px', 
                            'font-size': '24px', 
                            'text-align': 'left'
                        })
)

print("\nGenerating HTML and applying top-left Colspan Merge...")

# Convert to raw HTML string
html_str = final_styled_table.to_html()

# Inject CSS to force solid white backgrounds, fix the border gaps, AND draw the outer border
html_str = """
<style>
    body { background-color: #FFFFFF !important; margin: 0; padding: 20px; }
    table { 
        border-collapse: collapse !important; 
        background-color: #FFFFFF !important; 
        border: 3px solid black !important;   /* <--- Added the outer border here */
    }
    th, td { border-collapse: collapse !important; }
</style>
""" + html_str

# # HTML HACK: Find the two empty 'blank' cells generated by Pandas in the top left corner 
# # and replace them with a single merged header
# html_str = re.sub(
#     r'<th class="blank"[^>]*>.*?</th>\s*<th class="blank level0"[^>]*>.*?</th>',
#     r'<th colspan="2" style="text-align: center; vertical-align: bottom; font-weight: bold; font-size: 22px; border-bottom: 3px solid black; border-right: 1px solid #ccc; white-space: pre-wrap; padding-bottom: 10px;">Group/\nParameter</th>',
#     html_str,
#     count=1,
#     flags=re.IGNORECASE | re.DOTALL
# )

# HTML HACK: Find the THREE empty 'blank' cells generated by Pandas
html_str = re.sub(
    r'<th class="blank"[^>]*>.*?</th>\s*<th class="blank"[^>]*>.*?</th>\s*<th class="blank[^"]*"[^>]*>.*?</th>',
    r'<th colspan="3" style="text-align: center; vertical-align: bottom; font-weight: bold; font-size: 22px; border-bottom: 3px solid black; border-right: 1px solid #ccc; white-space: pre-wrap; padding-bottom: 10px;">Group/\nParameter</th>',
    html_str,
    count=1,
    flags=re.IGNORECASE | re.DOTALL
)

print("Saving final styled table as image...")

# Define export path
img_export_path = 'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/Comprehensive_Recovery_Metrics_Table.png'

# Export directly to PNG using Chrome natively
# dfi.export(final_styled_table, img_export_path, dpi=300, table_conversion='chrome') #old way before replacement

# Save the raw HTML to a temporary file
temp_html_path = 'temp_table.html'
with open(temp_html_path, 'w', encoding='utf-8') as f:
    f.write(html_str)

# Use html2image to render the file
# Set a large canvas to ensure the whole table fits before we crop it
hti = Html2Image(size=(1500, 2500)) 
temp_img_name = 'temp_render.png'
hti.screenshot(html_file=temp_html_path, save_as=temp_img_name)

# --- IMAGE PROCESSING: Fix Transparency and Auto-Crop ---
img = Image.open(temp_img_name)

# 1. Force white background (remove transparency)
if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
    background = Image.new('RGB', img.size, (255, 255, 255))
    background.paste(img, mask=img.split()[3]) # Paste using alpha channel as mask
    img = background
else:
    img = img.convert('RGB')

# 2. Auto-crop the image to the table boundaries
bg = Image.new('RGB', img.size, (255, 255, 255))
diff = ImageChops.difference(img, bg)
bbox = diff.getbbox() # Finds the bounding box of the non-white pixels

if bbox:
    # Add a 15-pixel white padding margin around the cropped table
    pad = 15
    bbox = (
        max(0, bbox[0] - pad), 
        max(0, bbox[1] - pad), 
        min(img.size[0], bbox[2] + pad), 
        min(img.size[1], bbox[3] + pad)
    )
    img = img.crop(bbox)

# Save the finalized, cropped, white-background image to your real folder
img.save(img_export_path)

# Cleanup temporary files
if os.path.exists(temp_html_path):
    os.remove(temp_html_path)
if os.path.exists(temp_img_name):
    os.remove(temp_img_name)

print(f"Table successfully saved and cropped to: {img_export_path}")




Calculating recovery metrics across all groups...

Generating HTML and applying top-left Colspan Merge...
Saving final styled table as image...
Table successfully saved and cropped to: G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/Comprehensive_Recovery_Metrics_Table.png


(OLD table 1)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Mapping requested group names to their exact file names and true generating parameters (from Table 1)
# Parameter order: [alphaRewards, gammaRewards, alphaPunishments, gammaPunishments, tau]
group_info = {
    "pre_ect":    {"file": "pre_ect",          "targets": [0.15, 0.70, 0.45, 0.23, 0.10]},
    "post_ect":   {"file": "post_ect",         "targets": [0.31, 0.54, 0.25, 0.25, 0.08]},
    "non_res_v1": {"file": "non_res_v1",       "targets": [0.12, 0.72, 0.28, 0.34, 0.08]},
    "non_res_v2": {"file": "non_res_v2",       "targets": [0.22, 0.81, 0.42, 0.15, 0.08]},
    "non_ect_v1": {"file": "non_ect_v1",       "targets": [0.21, 0.83, 0.46, 0.11, 0.17]},
    "non_ect_v2": {"file": "non_ect_v2",       "targets": [0.12, 0.79, 0.52, 0.15, 0.08]},
    "no_dep_v1":  {"file": "no_depression_v1", "targets": [0.16, 0.83, 0.53, 0.19, 0.11]},
    "no_dep_v2":  {"file": "no_depression_v2", "targets": [0.18, 0.73, 0.41, 0.23, 0.07]},
}

variables = ["learnrate_pos", "discount_pos", "learnrate_neg", "discount_neg", "tau"]
num_subjects = 50
results_list = []

print("Calculating recovery metrics across all groups...")

for group_name, info in group_info.items():
    file_path = f'G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/modeled_data_VPRL_{info["file"]}.csv'
    
    try:
        # Load the summary dataframe, using the first column (parameter names) as the index
        df = pd.read_csv(file_path, index_col=0)
        # Calculate total divergences for this group's fit
        total_divergences = int(np.round(df.loc['divergent__', 'mean'] * df.loc['divergent__', 'count']))
    except FileNotFoundError:
        print(f"Warning: Could not find data for {group_name} at {file_path}. Skipping.")
        continue

    for col_idx, var in enumerate(variables):
        true_target = info["targets"][col_idx]
        
        # Group-level standard deviation (used as the prior/reference for shrinkage and KL)
        group_sd = df.loc[f'mu_{var}', 'std']
        group_mu = df.loc[f'mu_{var}', 'mean']
        
        agent_shrinkages = []
        agent_zscores = []
        agent_pvals = []
        agent_kls = []
        
        for x in range(1, num_subjects + 1):
            sim_key = f"{var}[{x}]"
            
            if sim_key in df.index:
                agent_mean = df.loc[sim_key, 'mean']
                agent_median = df.loc[sim_key, 'median']
                agent_sd = df.loc[sim_key, 'std']
                
                # 1. Posterior Shrinkage
                shrinkage = 1 - ( (agent_sd**2) / (group_sd**2) )
                agent_shrinkages.append(shrinkage)
                
                # 2. KL Divergence (Agent vs Group Posterior)
                kl_div = np.log(group_sd / agent_sd) + ((agent_sd**2 + (agent_mean - group_mu)**2) / (2 * group_sd**2)) - 0.5
                agent_kls.append(kl_div)
                
                # 3. Z-Score (Agent Mean vs True Target)
                z_score = np.abs(agent_mean - true_target) / agent_sd
                # z_score = np.abs(agent_median - true_target) / agent_sd
                agent_zscores.append(z_score)
                
                # 4. P-value (Two-tailed test from Z-score)
                p_val = 2 * norm.sf(z_score)
                agent_pvals.append(p_val)
                
        # Aggregate metrics for this specific parameter in this specific group
        results_list.append({
            "Group": group_name,
            "Parameter": var,
            "Total_Divergences": total_divergences,
            "Mean_Shrinkage": np.mean(agent_shrinkages),
            "Mean_KL_Div": np.mean(agent_kls),
            "Mean_Z_Score": np.mean(agent_zscores),
            "Mean_P_Value": np.mean(agent_pvals),
            "Pct_Sig_Recovery": np.mean(np.array(agent_pvals) > 0.05) * 100 # % of agents whose recovered mean is NOT significantly different from the true target
        })

# =========================================================================
# Create DataFrame and Format with Merged, Rotated Group Cells
# =========================================================================
metrics_df = pd.DataFrame(results_list)

# 1. Set Group and Parameter as a MultiIndex to automatically vertical-merge Group cells
styled_df = metrics_df.set_index(['Group', 'Parameter'])

# 2. Define the formatting rules for data columns
format_dict = {
    'Mean_Shrinkage': '{:.3f}', 
    'Mean_KL_Div': '{:.3f}',
    'Mean_Z_Score': '{:.3f}', 
    'Mean_P_Value': '{:.4f}',
    'Pct_Sig_Recovery': '{:.1f}%'
}

# 3. Define custom CSS to rotate the outer index (level0 / Group) column
rotation_style = [
    {
        'selector': '.row_heading.level0',
        'props': [
            ('writing-mode', 'vertical-rl'),
            ('transform', 'rotate(180deg)'),  # Rotates text to read cleanly bottom-to-top
            ('text-align', 'center'),
            ('vertical-align', 'middle'),
            ('font-weight', 'bold'),
            ('background-color', '#f7f7f9'), # Subtle background color for the group block
            ('padding', '8px')
        ]
    },
    {
        'selector': '.row_heading.level1', # Keeps parameter names left-aligned or centered cleanly
        'props': [
            ('text-align', 'left'),
            ('padding', '8px')
        ]
    }
]

print("\nRecovery Metrics Table:")
# Render the formatted table with custom vertical styling
display(styled_df.style.format(format_dict)
                       .set_table_styles(rotation_style)
                       .set_properties(**{'text-align': 'center', 'vertical-align': 'middle'}))

# Optional: Save the raw unstyled dataframe to CSV if needed
metrics_df.to_csv('G:/My Drive/Research/Code/STAN Parameter Recovery/Groups/Comprehensive_Recovery_Metrics.csv', index=False)